# Refinamento do Modelo de Risco de Crédito — Análise de Threshold e Custo Financeiro

## Contexto e motivação

O notebook anterior (**CatBoost + Optuna**) encerrou com a seguinte conclusão:

> *"O modelo otimizado é mais forte, mas precisa de ajuste de threshold para ficar melhor alinhado à estratégia de risco de crédito. A próxima etapa recomendada é construir uma tabela de decisão por threshold, avaliando precision, recall, F1, F2, falsos positivos, falsos negativos e impacto financeiro estimado em cada ponto de corte."*

### Problema identificado

Com threshold padrão de **0.50**, o CatBoost + Optuna produziu:

| Efeito | Valor |
|---|---:|
| Precision | **0.9767** ✅ muito alta |
| Recall | **0.7405** ⚠️ queda relevante vs baseline (0.7870) |
| Falsos Negativos | **368** (66 a mais que o baseline) |
| Falsos Positivos | **25** (154 a menos que o baseline) |

O modelo ficou **muito conservador**: raramente classifica como risco alto, acertando quase sempre quando o faz, mas deixando passar clientes inadimplentes.

### O que este notebook faz

1. Reproduz o pipeline completo (dados → modelo → predições)
2. Constrói a tabela de decisão por threshold com todas as métricas
3. Incorpora **análise de custo financeiro** com matriz de custo configurável
4. Identifica thresholds ótimos para diferentes objetivos de negócio
5. Gera o modelo final re-treinado com o threshold escolhido
6. Compara todos os cenários em tabela e gráficos
7. Produz recomendação de negócio fundamentada nos resultados

---
> **Como usar:** coloque o arquivo `credit_risk_dataset.csv` na mesma pasta do notebook (ou ajuste `DATA_PATH`) e execute as células em sequência.

## 1. Instalação e imports

In [ ]:
import sys, subprocess, importlib.util

required = {
    'catboost': 'catboost',
    'optuna':   'optuna',
    'shap':     'shap',
    'sklearn':  'scikit-learn',
}
for mod, pkg in required.items():
    if importlib.util.find_spec(mod) is None:
        print(f'Instalando {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('Ambiente pronto.')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score,
    log_loss, brier_score_loss,
    matthews_corrcoef, cohen_kappa_score,
    confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay,
)
from sklearn.calibration import calibration_curve

from catboost import CatBoostClassifier, Pool
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

print('✅ Imports concluídos.')

## 2. Carregamento e preparação dos dados

In [ ]:
# ── Ajuste DATA_PATH se necessário ──────────────────────────────────────────
DATA_PATH = Path('credit_risk_dataset.csv')

for alt in [
    Path('/mnt/data/credit_risk_dataset(1).csv'),
    Path('credit_risk_dataset(1).csv'),
]:
    if not DATA_PATH.exists() and alt.exists():
        DATA_PATH = alt

if not DATA_PATH.exists():
    # Tenta upload no Colab
    try:
        from google.colab import files
        print('Faça upload do credit_risk_dataset.csv')
        up = files.upload()
        DATA_PATH = Path(list(up.keys())[0])
    except Exception:
        raise FileNotFoundError('credit_risk_dataset.csv não encontrado.')

df_raw = pd.read_csv(DATA_PATH)
print(f'Linhas: {df_raw.shape[0]:,}  |  Colunas: {df_raw.shape[1]:,}')
df_raw.head(3)

In [ ]:
TARGET = 'loan_status'

model_df = df_raw.drop_duplicates().reset_index(drop=True)

cat_cols = model_df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
num_cols = model_df.select_dtypes(include=['number']).columns.tolist()
if TARGET in num_cols:  num_cols.remove(TARGET)
if TARGET in cat_cols:  cat_cols.remove(TARGET)

for col in cat_cols:
    model_df[col] = model_df[col].astype('string').fillna('__MISSING__')

X = model_df.drop(columns=[TARGET])
y = model_df[TARGET].astype(int)

cat_features = [X.columns.get_loc(c) for c in cat_cols]

print(f'Shape X: {X.shape}   |   Taxa de evento: {y.mean():.2%}')
print(f'Categóricas: {cat_cols}')
print(f'Numéricas: {num_cols}')

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.20, random_state=RANDOM_STATE, stratify=y_train_full
)

print(f'Treino:    {X_train.shape}  taxa={y_train.mean():.2%}')
print(f'Validação: {X_valid.shape}  taxa={y_valid.mean():.2%}')
print(f'Teste:     {X_test.shape}   taxa={y_test.mean():.2%}')

## 3. Reprodução do modelo CatBoost + Optuna

Reproduzimos o pipeline do notebook anterior para obter as probabilidades na base de teste.

In [ ]:
def ks_statistic(y_true, y_proba):
    data = pd.DataFrame({'y': y_true, 'p': y_proba}).sort_values('p', ascending=False)
    pos = (data['y'] == 1).sum()
    neg = (data['y'] == 0).sum()
    if pos == 0 or neg == 0: return np.nan
    data['cp'] = (data['y'] == 1).cumsum() / pos
    data['cn'] = (data['y'] == 0).cumsum() / neg
    return (data['cp'] - data['cn']).abs().max()


def objective(trial):
    params = {
        'iterations':       trial.suggest_int('iterations', 300, 1500),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'depth':            trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg':      trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength':  trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
        'border_count':     trial.suggest_int('border_count', 32, 255),
        'loss_function':    'Logloss',
        'eval_metric':      'AUC',
        'auto_class_weights': 'Balanced',
        'random_seed':      RANDOM_STATE,
        'verbose':          False,
    }
    model = CatBoostClassifier(**params)
    model.fit(
        X_train, y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=100,
        use_best_model=True,
    )
    proba = model.predict_proba(X_valid)[:, 1]
    return roc_auc_score(y_valid, proba)


print('Iniciando otimização com Optuna (50 trials)...')
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\n✅ Melhor ROC-AUC na validação: {study.best_value:.4f}')
print('Melhores hiperparâmetros:')
best_params = study.best_params
for k, v in best_params.items():
    print(f'  {k}: {v}')

In [ ]:
final_params = {
    **best_params,
    'loss_function':      'Logloss',
    'eval_metric':        'AUC',
    'auto_class_weights': 'Balanced',
    'random_seed':        RANDOM_STATE,
    'verbose':            100,
}

optimized_model = CatBoostClassifier(**final_params)
optimized_model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    early_stopping_rounds=100,
    use_best_model=True,
)

y_proba = optimized_model.predict_proba(X_test)[:, 1]
print('\n✅ Modelo treinado. Probabilidades geradas para o conjunto de teste.')

## 4. Tabela de decisão por threshold

Varremos thresholds de 0.05 a 0.75 e calculamos todas as métricas relevantes,
permitindo visualizar com precisão o trade-off entre captura de risco e alarmes falsos.

In [ ]:
thresholds = np.round(np.arange(0.05, 0.76, 0.01), 2)
rows = []

for th in thresholds:
    pred = (y_proba >= th).astype(int)
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()
    precision = precision_score(y_test, pred, zero_division=0)
    recall    = recall_score(y_test, pred, zero_division=0)
    spec      = tn / (tn + fp) if (tn + fp) else np.nan
    rows.append({
        'threshold':          round(th, 2),
        'accuracy':           accuracy_score(y_test, pred),
        'balanced_accuracy':  balanced_accuracy_score(y_test, pred),
        'precision':          precision,
        'recall':             recall,
        'specificity':        spec,
        'f1':                 f1_score(y_test, pred, zero_division=0),
        'f2':                 fbeta_score(y_test, pred, beta=2, zero_division=0),
        'mcc':                matthews_corrcoef(y_test, pred),
        'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn),
    })

th_df = pd.DataFrame(rows)

# Thresholds de referência do notebook anterior
ref_thresholds = [0.30, 0.35, 0.40, 0.45, 0.50]
print('=== Thresholds de referência (recomendados na conclusão anterior) ===')
display(th_df[th_df['threshold'].isin(ref_thresholds)].set_index('threshold'))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Métricas principais
ax = axes[0, 0]
for metric, color in zip(
    ['precision', 'recall', 'f1', 'f2', 'balanced_accuracy', 'mcc'],
    ['royalblue', 'tomato', 'green', 'darkorange', 'purple', 'brown']
):
    ax.plot(th_df['threshold'], th_df[metric], label=metric, color=color)
for th_ref in [0.40, 0.45, 0.50]:
    ax.axvline(th_ref, linestyle='--', alpha=0.4, color='gray')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Métricas de classificação por threshold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 2: FN e FP absolutos
ax2 = axes[0, 1]
ax2.plot(th_df['threshold'], th_df['fn'], label='Falsos Negativos (risco perdido)', color='tomato', linewidth=2)
ax2.plot(th_df['threshold'], th_df['fp'], label='Falsos Positivos (bons reprovados)', color='royalblue', linewidth=2)
for th_ref in [0.40, 0.45, 0.50]:
    ax2.axvline(th_ref, linestyle='--', alpha=0.4, color='gray')
ax2.axhline(302, linestyle=':', color='salmon', label='FN baseline (302)')
ax2.axhline(368, linestyle=':', color='darkred', label='FN Optuna th=0.50 (368)')
ax2.set_xlabel('Threshold')
ax2.set_ylabel('Quantidade')
ax2.set_title('Falsos Negativos e Positivos absolutos por threshold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# Plot 3: Precision × Recall
ax3 = axes[1, 0]
ax3.plot(th_df['recall'], th_df['precision'], marker='o', markersize=3, color='navy')
for _, row in th_df[th_df['threshold'].isin([0.30, 0.35, 0.40, 0.45, 0.50])].iterrows():
    ax3.annotate(f"th={row['threshold']}",
                 (row['recall'], row['precision']),
                 textcoords='offset points', xytext=(5, 5), fontsize=8)
ax3.set_xlabel('Recall')
ax3.set_ylabel('Precision')
ax3.set_title('Curva Precision × Recall (por threshold)')
ax3.grid(True, alpha=0.3)

# Plot 4: F1 e F2 zoom
ax4 = axes[1, 1]
ax4.plot(th_df['threshold'], th_df['f1'], label='F1 (recall=precision)', color='green', linewidth=2)
ax4.plot(th_df['threshold'], th_df['f2'], label='F2 (recall 2x mais importante)', color='darkorange', linewidth=2)
best_f1_th = th_df.loc[th_df['f1'].idxmax(), 'threshold']
best_f2_th = th_df.loc[th_df['f2'].idxmax(), 'threshold']
ax4.axvline(best_f1_th, linestyle='--', color='green', alpha=0.5, label=f'Ótimo F1 (th={best_f1_th})')
ax4.axvline(best_f2_th, linestyle='--', color='darkorange', alpha=0.5, label=f'Ótimo F2 (th={best_f2_th})')
ax4.set_xlabel('Threshold')
ax4.set_ylabel('Score')
ax4.set_title('F1 e F2 por threshold')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)

plt.suptitle('Análise de Threshold — CatBoost + Optuna', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('threshold_analysis_refinamento.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Threshold ótimo por F1: {best_f1_th}')
print(f'Threshold ótimo por F2: {best_f2_th}')

## 5. Análise de custo financeiro por threshold

### Lógica da matriz de custo

Em risco de crédito, cada célula da matriz de confusão tem um impacto financeiro diferente:

| Situação | Impacto |
|---|---|
| **TP** — cliente de risco corretamente rejeitado | Evita perda de inadimplência |
| **FN** — cliente de risco aprovado indevidamente | **Perda** = valor do empréstimo × taxa de inadimplência |
| **FP** — cliente bom rejeitado indevidamente | **Perda** = receita de juros não realizada (custo de oportunidade) |
| **TN** — cliente bom corretamente aprovado | Receita normal |

### Parâmetros configuráveis

Ajuste os valores abaixo conforme o portfólio real:

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# PARÂMETROS DA MATRIZ DE CUSTO — ajuste conforme o negócio
# ════════════════════════════════════════════════════════════════════════════

# Custo de um Falso Negativo: cliente inadimplente aprovado
# Estimativa: valor médio do empréstimo que não será recuperado
CUSTO_FN = 5_000   # R$ 5.000 de perda por cliente inadimplente não detectado

# Custo de um Falso Positivo: cliente bom rejeitado
# Estimativa: receita de juros não realizada (custo de oportunidade)
CUSTO_FP = 800     # R$ 800 de receita perdida por bom cliente rejeitado

# Cenários adicionais para análise de sensibilidade
# (razão custo_FN / custo_FP)
CENARIOS = [
    {'nome': 'Conservador (FN=10× FP)',    'custo_fn': 8_000, 'custo_fp': 800},
    {'nome': 'Base (FN=6.25× FP)',         'custo_fn': 5_000, 'custo_fp': 800},
    {'nome': 'Moderado (FN=3× FP)',        'custo_fn': 2_400, 'custo_fp': 800},
    {'nome': 'Balanceado (FN=1× FP)',      'custo_fn': 800,   'custo_fp': 800},
]
# ════════════════════════════════════════════════════════════════════════════

print('Parâmetros configurados:')
print(f'  Custo por FN (inadimplente não detectado): R$ {CUSTO_FN:,.0f}')
print(f'  Custo por FP (bom cliente rejeitado):      R$ {CUSTO_FP:,.0f}')
print(f'  Razão FN/FP: {CUSTO_FN/CUSTO_FP:.2f}×')

In [ ]:
# Calcula custo total e por cliente para cada threshold
th_df['custo_total'] = th_df['fn'] * CUSTO_FN + th_df['fp'] * CUSTO_FP
th_df['custo_fn_total'] = th_df['fn'] * CUSTO_FN
th_df['custo_fp_total'] = th_df['fp'] * CUSTO_FP
th_df['custo_por_cliente'] = th_df['custo_total'] / len(y_test)

# Threshold de menor custo total
best_cost_th   = th_df.loc[th_df['custo_total'].idxmin(), 'threshold']
best_cost_row  = th_df.loc[th_df['custo_total'].idxmin()]
baseline_cost  = 302 * CUSTO_FN + 179 * CUSTO_FP  # custo do baseline (notebook anterior)
optuna50_cost  = 368 * CUSTO_FN + 25  * CUSTO_FP  # custo do Optuna th=0.50

print(f'Custo estimado — Baseline th=0.50:          R$ {baseline_cost:>12,.0f}')
print(f'Custo estimado — Optuna th=0.50:            R$ {optuna50_cost:>12,.0f}')
print(f'Custo estimado — Optuna th={best_cost_th}:  R$ {best_cost_row["custo_total"]:>12,.0f}')
print()
print(f'Economia vs Optuna th=0.50:  R$ {optuna50_cost - best_cost_row["custo_total"]:,.0f}')
print(f'Economia vs Baseline th=0.50: R$ {baseline_cost - best_cost_row["custo_total"]:,.0f}')
print(f'\n➡️  Threshold de menor custo: {best_cost_th}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Custo total por threshold
ax = axes[0]
ax.plot(th_df['threshold'], th_df['custo_total'] / 1_000, color='navy', linewidth=2)
ax.fill_between(th_df['threshold'], th_df['custo_fn_total'] / 1_000, alpha=0.3,
                color='tomato', label=f'Custo FN (inadimplência perdida)')
ax.fill_between(th_df['threshold'], th_df['custo_fp_total'] / 1_000, alpha=0.3,
                color='steelblue', label=f'Custo FP (bom cliente rejeitado)')
ax.axvline(best_cost_th, linestyle='--', color='green', linewidth=2,
           label=f'Menor custo (th={best_cost_th})')
ax.axvline(0.50, linestyle=':', color='gray', linewidth=1.5, label='th=0.50 (padrão)')
ax.axhline(optuna50_cost / 1_000, linestyle=':', color='orange', label='Optuna th=0.50')
ax.axhline(baseline_cost / 1_000, linestyle=':', color='red', label='Baseline th=0.50')
ax.set_xlabel('Threshold')
ax.set_ylabel('Custo total (R$ mil)')
ax.set_title('Custo financeiro estimado por threshold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}k'))

# Plot 2: Sensibilidade por cenário
ax2 = axes[1]
for cenario in CENARIOS:
    custo = th_df['fn'] * cenario['custo_fn'] + th_df['fp'] * cenario['custo_fp']
    ax2.plot(th_df['threshold'], custo / 1_000, label=cenario['nome'], linewidth=1.8)
ax2.axvline(0.50, linestyle=':', color='gray', linewidth=1.5, label='th=0.50')
ax2.set_xlabel('Threshold')
ax2.set_ylabel('Custo total (R$ mil)')
ax2.set_title('Análise de sensibilidade — custo por cenário')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}k'))

plt.suptitle('Análise de Custo Financeiro por Threshold', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('custo_financeiro_threshold.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Identificação dos thresholds ótimos para cada objetivo

Consolidamos os melhores thresholds segundo cada critério para apoiar a decisão de negócio.

In [ ]:
# Melhor por cada critério
criterios = {
    'Menor custo financeiro':       th_df.loc[th_df['custo_total'].idxmin()],
    'Maior F2 (favorece recall)':   th_df.loc[th_df['f2'].idxmax()],
    'Maior F1 (equilibrado)':       th_df.loc[th_df['f1'].idxmax()],
    'Maior MCC (robusto)':          th_df.loc[th_df['mcc'].idxmax()],
    'Maior Balanced Accuracy':      th_df.loc[th_df['balanced_accuracy'].idxmax()],
    'Recall ≥ 0.80 + max precision': th_df[th_df['recall'] >= 0.80].loc[
                                         th_df[th_df['recall'] >= 0.80]['precision'].idxmax()
                                     ] if (th_df['recall'] >= 0.80).any() else th_df.iloc[0],
    'Padrão Optuna (th=0.50)':      th_df[th_df['threshold'] == 0.50].iloc[0],
}

summary_rows = []
for criterio, row in criterios.items():
    summary_rows.append({
        'Critério':    criterio,
        'Threshold':   row['threshold'],
        'Precision':   row['precision'],
        'Recall':      row['recall'],
        'F1':          row['f1'],
        'F2':          row['f2'],
        'MCC':         row['mcc'],
        'FN':          int(row['fn']),
        'FP':          int(row['fp']),
        'Custo (R$)':  int(row['custo_total']),
    })

summary_df = pd.DataFrame(summary_rows).set_index('Critério')
print('=== Resumo dos thresholds ótimos por critério ===')
display(summary_df.style.format({
    'Threshold': '{:.2f}',
    'Precision': '{:.4f}',
    'Recall':    '{:.4f}',
    'F1':        '{:.4f}',
    'F2':        '{:.4f}',
    'MCC':       '{:.4f}',
    'Custo (R$)': 'R$ {:,.0f}',
}).highlight_min(subset=['FN', 'Custo (R$)'], color='#d4edda'
).highlight_max(subset=['Recall', 'F2'], color='#cce5ff'))

## 7. Avaliação completa dos cenários escolhidos

Avaliamos em detalhe os três thresholds mais relevantes para a decisão:
- **Threshold padrão (0.50)** — referência do notebook anterior  
- **Threshold de menor custo** — minimiza impacto financeiro  
- **Threshold ótimo por F2** — maximiza captura de inadimplentes

In [ ]:
def evaluate_full(y_true, y_proba_arr, threshold, label):
    pred = (y_proba_arr >= threshold).astype(int)
    cm   = confusion_matrix(y_true, pred)
    tn, fp, fn, tp = cm.ravel()
    prec = precision_score(y_true, pred, zero_division=0)
    rec  = recall_score(y_true, pred, zero_division=0)
    roc  = roc_auc_score(y_true, y_proba_arr)
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    return {
        'Modelo':           label,
        'Threshold':        threshold,
        'Accuracy':         accuracy_score(y_true, pred),
        'Balanced Acc.':    balanced_accuracy_score(y_true, pred),
        'Precision':        prec,
        'Recall':           rec,
        'Specificity':      spec,
        'F1':               f1_score(y_true, pred, zero_division=0),
        'F2':               fbeta_score(y_true, pred, beta=2, zero_division=0),
        'ROC AUC':          roc,
        'PR AUC':           average_precision_score(y_true, y_proba_arr),
        'Log Loss':         log_loss(y_true, y_proba_arr),
        'Brier':            brier_score_loss(y_true, y_proba_arr),
        'MCC':              matthews_corrcoef(y_true, pred),
        'Kappa':            cohen_kappa_score(y_true, pred),
        'KS':               ks_statistic(y_true, y_proba_arr),
        'Gini':             2 * roc - 1,
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
        'Custo Total':      fn * CUSTO_FN + fp * CUSTO_FP,
    }


CHOSEN_THRESHOLDS = {
    f'Optuna th=0.50 (padrão anterior)':     0.50,
    f'Optuna th={best_f2_th} (máx. F2)':     best_f2_th,
    f'Optuna th={best_cost_th} (menor custo)': best_cost_th,
}

eval_results = []
for label, th in CHOSEN_THRESHOLDS.items():
    eval_results.append(evaluate_full(y_test, y_proba, th, label))

eval_df = pd.DataFrame(eval_results).set_index('Modelo')

print('=== Comparativo completo entre thresholds escolhidos ===')
display(eval_df.T.style.format({
    row: '{:.4f}' for row in eval_df.columns if eval_df[row].dtype == float
}))

In [ ]:
# Gráficos completos para o threshold de menor custo (escolha recomendada)
FINAL_TH = best_cost_th
final_pred = (y_proba >= FINAL_TH).astype(int)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ROC
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[0, 0])
axes[0, 0].set_title(f'Curva ROC — th={FINAL_TH}')

# PR
PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=axes[0, 1])
axes[0, 1].axhline(y_test.mean(), color='r', linestyle='--', label=f'Prevalência ({y_test.mean():.2%})')
axes[0, 1].set_title(f'Curva Precision-Recall — th={FINAL_TH}')
axes[0, 1].legend()

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, final_pred, ax=axes[0, 2], cmap='Blues', values_format='d',
    display_labels=['Classe 0\n(bom pagador)', 'Classe 1\n(risco/default)']
)
axes[0, 2].set_title(f'Matriz de Confusão — th={FINAL_TH}')

# Calibração
prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10, strategy='quantile')
axes[1, 0].plot(prob_pred, prob_true, marker='o', label='Modelo')
axes[1, 0].plot([0, 1], [0, 1], '--', label='Calibração perfeita')
axes[1, 0].set_xlabel('Probabilidade prevista')
axes[1, 0].set_ylabel('Frequência real')
axes[1, 0].set_title('Curva de calibração')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Distribuição dos scores por classe
axes[1, 1].hist(y_proba[y_test == 0], bins=50, alpha=0.6, color='steelblue', label='Classe 0 (bom pagador)', density=True)
axes[1, 1].hist(y_proba[y_test == 1], bins=50, alpha=0.6, color='tomato', label='Classe 1 (risco/default)', density=True)
axes[1, 1].axvline(FINAL_TH, color='black', linestyle='--', linewidth=2, label=f'Threshold={FINAL_TH}')
axes[1, 1].axvline(0.50, color='gray', linestyle=':', linewidth=1.5, label='Threshold=0.50')
axes[1, 1].set_xlabel('Score de probabilidade')
axes[1, 1].set_ylabel('Densidade')
axes[1, 1].set_title('Distribuição dos scores por classe')
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(True, alpha=0.3)

# Custo acumulado com comparativo
thresholds_plot = [0.30, 0.35, 0.40, 0.45, 0.50, FINAL_TH, best_f2_th]
custos_plot = [
    th_df[th_df['threshold'] == th]['custo_total'].values[0]
    for th in sorted(set(thresholds_plot))
]
labels_plot = [str(th) for th in sorted(set(thresholds_plot))]
colors_bar = ['tomato' if th == 0.50 else 'green' if th == FINAL_TH else 'steelblue'
              for th in sorted(set(thresholds_plot))]
axes[1, 2].bar(labels_plot, [c / 1000 for c in custos_plot], color=colors_bar, edgecolor='black')
axes[1, 2].set_xlabel('Threshold')
axes[1, 2].set_ylabel('Custo total (R$ mil)')
axes[1, 2].set_title('Comparativo de custo por threshold')
axes[1, 2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}k'))
axes[1, 2].grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Avaliação Detalhada — CatBoost + Optuna  |  Threshold escolhido: {FINAL_TH}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('avaliacao_threshold_final.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nClassification report (th={FINAL_TH}):')
print(classification_report(y_test, final_pred, digits=4,
                            target_names=['Classe 0 (bom)', 'Classe 1 (risco)']))

## 8. Curvas de ganho e lift — comparativo entre thresholds

In [ ]:
def gains_lift_table(y_true, proba, n_bins=10):
    data = pd.DataFrame({
        'y': y_true.values if hasattr(y_true, 'values') else y_true,
        'proba': proba
    }).sort_values('proba', ascending=False).reset_index(drop=True)
    data['decile'] = pd.qcut(data.index + 1, q=n_bins, labels=False) + 1
    base_rate    = data['y'].mean()
    total_events = data['y'].sum()
    table = data.groupby('decile').agg(
        qtd=('y', 'count'), eventos=('y', 'sum'),
        score_min=('proba', 'min'), score_max=('proba', 'max'),
        taxa_evento=('y', 'mean'),
    ).reset_index()
    table['eventos_acumulados']   = table['eventos'].cumsum()
    table['ganho_acumulado_%']    = table['eventos_acumulados'] / total_events * 100
    table['populacao_acumulada_%'] = table['qtd'].cumsum() / len(data) * 100
    table['lift']                 = table['taxa_evento'] / base_rate
    table['lift_acumulado']       = (table['eventos_acumulados'] / table['qtd'].cumsum()) / base_rate
    return table

gl = gains_lift_table(y_test, y_proba)

print('=== Tabela de ganhos e lift — CatBoost + Optuna ===')
display(gl)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(gl['populacao_acumulada_%'], gl['ganho_acumulado_%'], marker='o', label='Modelo', color='navy')
axes[0].plot([0, 100], [0, 100], linestyle='--', color='gray', label='Aleatório')
axes[0].fill_between(gl['populacao_acumulada_%'], gl['ganho_acumulado_%'],
                      gl['populacao_acumulada_%'], alpha=0.1, color='navy', label='Ganho sobre aleatório')
axes[0].set_xlabel('População acumulada (%)')
axes[0].set_ylabel('Defaults capturados (%)')
axes[0].set_title('Curva de Ganhos Acumulados')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

sns.barplot(data=gl, x='decile', y='lift', ax=axes[1],
            palette=['tomato' if i < 3 else 'steelblue' for i in range(len(gl))])
axes[1].axhline(1.0, linestyle='--', color='gray', label='Lift = 1 (sem ganho)')
axes[1].set_xlabel('Decil (1 = maior risco previsto)')
axes[1].set_ylabel('Lift')
axes[1].set_title('Lift por decil de risco')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# Anotações nos decis superiores
for i, row in gl.iterrows():
    if row['decile'] <= 3:
        axes[1].text(i, row['lift'] + 0.05, f"{row['lift']:.2f}×",
                     ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Curvas de Ganho e Lift — CatBoost + Optuna', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('gains_lift_refinamento.png', dpi=120, bbox_inches='tight')
plt.show()

top3 = gl[gl['decile'] <= 3]
print(f'\n📌 Nos 30% mais arriscados, o modelo captura {top3["ganho_acumulado_%"].iloc[-1]:.1f}% dos defaults.')
print(f'📌 Lift médio nos 3 primeiros decis: {top3["lift"].mean():.2f}×')

## 9. Importância das variáveis e interpretação SHAP

In [ ]:
feature_importance = pd.DataFrame({
    'feature':    X.columns,
    'importance': optimized_model.get_feature_importance(),
}).sort_values('importance', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d62728' if i < 3 else '#1f77b4' for i in range(len(feature_importance))]
ax.barh(feature_importance['feature'][::-1], feature_importance['importance'][::-1], color=colors[::-1])
ax.set_xlabel('Importância (CatBoost)')
ax.set_title('Importância das variáveis — CatBoost + Optuna')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('feature_importance_refinamento.png', dpi=120, bbox_inches='tight')
plt.show()

display(feature_importance)

In [ ]:
try:
    import shap
    shap.initjs()

    sample_size = min(2000, len(X_test))
    X_shap = X_test.sample(sample_size, random_state=RANDOM_STATE)

    shap_values = optimized_model.get_feature_importance(
        Pool(X_shap, cat_features=cat_features),
        type='ShapValues'
    )
    shap_values_only = shap_values[:, :-1]  # remove bias term

    print('=== SHAP — Importância global ===')
    shap.summary_plot(shap_values_only, X_shap, plot_type='bar', show=True)

    print('\n=== SHAP — Direção e magnitude do impacto ===')
    shap.summary_plot(shap_values_only, X_shap, show=True)

    # Dependence plot para a feature mais importante
    top_feat = feature_importance.iloc[0]['feature']
    print(f'\n=== SHAP Dependence plot — {top_feat} ===')
    shap.dependence_plot(top_feat, shap_values_only, X_shap, show=True)

except Exception as e:
    print(f'SHAP não disponível neste ambiente: {e}')

## 10. Validação cruzada com o threshold refinado

Verificamos se o ganho obtido com o novo threshold é estável entre diferentes divisões dos dados.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train_full, y_train_full), start=1):
    X_tr, X_va = X_train_full.iloc[tr_idx], X_train_full.iloc[va_idx]
    y_tr, y_va = y_train_full.iloc[tr_idx], y_train_full.iloc[va_idx]

    fold_model = CatBoostClassifier(**{**final_params, 'verbose': False})
    fold_model.fit(
        X_tr, y_tr,
        cat_features=cat_features,
        eval_set=(X_va, y_va),
        early_stopping_rounds=100,
        use_best_model=True,
    )
    proba_va = fold_model.predict_proba(X_va)[:, 1]

    # Avalia nos dois thresholds: padrão e refinado
    for th_label, th in [('th=0.50', 0.50), (f'th={FINAL_TH}', FINAL_TH)]:
        pred_va = (proba_va >= th).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_va, pred_va).ravel()
        cv_scores.append({
            'fold': fold,
            'threshold': th_label,
            'roc_auc':   roc_auc_score(y_va, proba_va),
            'pr_auc':    average_precision_score(y_va, proba_va),
            'f1':        f1_score(y_va, pred_va, zero_division=0),
            'f2':        fbeta_score(y_va, pred_va, beta=2, zero_division=0),
            'recall':    recall_score(y_va, pred_va, zero_division=0),
            'precision': precision_score(y_va, pred_va, zero_division=0),
            'fn':        int(fn),
            'fp':        int(fp),
            'custo':     fn * CUSTO_FN + fp * CUSTO_FP,
        })

cv_df = pd.DataFrame(cv_scores)

print('=== Validação cruzada — th=0.50 vs th refinado ===')
for th_label in cv_df['threshold'].unique():
    subset = cv_df[cv_df['threshold'] == th_label].drop(columns=['fold', 'threshold'])
    print(f'\n--- {th_label} ---')
    display(subset.agg(['mean', 'std']))

## 11. Tabela final comparativa — todos os modelos e thresholds

In [ ]:
# Inclui resultados do notebook anterior (baseline)
# Os valores abaixo são os reportados na síntese final
BASELINE_METRICS = {
    'Modelo':        'CatBoost Baseline th=0.50',
    'Threshold':      0.50,
    'Accuracy':       0.9258,
    'Balanced Acc.':  0.8758,
    'Precision':      0.8618,
    'Recall':         0.7870,
    'Specificity':    0.9647,
    'F1':             0.8227,
    'F2':             0.8009,
    'ROC AUC':        0.9436,
    'PR AUC':         0.9016,
    'Log Loss':       0.2217,
    'Brier':          0.0632,
    'MCC':            0.7772,
    'Kappa':          0.7759,
    'KS':             0.7561,
    'Gini':           0.8871,
    'TN': 4887, 'FP': 179, 'FN': 302, 'TP': 1116,
    'Custo Total':    302 * CUSTO_FN + 179 * CUSTO_FP,
}

# Métricas do modelo Optuna na versão atual
all_eval = [BASELINE_METRICS] + eval_results
final_compare = pd.DataFrame(all_eval).set_index('Modelo')

cols_display = ['Threshold', 'Precision', 'Recall', 'F1', 'F2', 'MCC',
                'ROC AUC', 'PR AUC', 'FN', 'FP', 'Custo Total']

print('=== TABELA FINAL — Comparativo completo ===')
display(
    final_compare[cols_display].style
    .format({'Threshold': '{:.2f}', 'Precision': '{:.4f}', 'Recall': '{:.4f}',
             'F1': '{:.4f}', 'F2': '{:.4f}', 'MCC': '{:.4f}',
             'ROC AUC': '{:.4f}', 'PR AUC': '{:.4f}',
             'FN': '{:d}', 'FP': '{:d}', 'Custo Total': 'R$ {:,.0f}'})
    .highlight_min(subset=['FN', 'FP', 'Custo Total'], color='#d4edda')
    .highlight_max(subset=['Recall', 'F2', 'F1', 'MCC'], color='#cce5ff')
)

## 12. Exportação dos resultados

In [ ]:
OUTPUT_DIR = Path('outputs_refinamento')
OUTPUT_DIR.mkdir(exist_ok=True)

# Tabela de thresholds completa
th_df.to_csv(OUTPUT_DIR / 'threshold_decision_table.csv', index=False)

# Predições com os dois thresholds
predictions_df = X_test.copy()
predictions_df['y_true']                   = y_test.values
predictions_df['proba_default']            = y_proba
predictions_df['pred_th_0_50']             = (y_proba >= 0.50).astype(int)
predictions_df[f'pred_th_{FINAL_TH}']      = (y_proba >= FINAL_TH).astype(int)
predictions_df[f'pred_th_{best_f2_th}_f2'] = (y_proba >= best_f2_th).astype(int)
predictions_df.to_csv(OUTPUT_DIR / 'credit_risk_predictions_refinamento.csv', index=False)

# Comparativo final
final_compare.to_csv(OUTPUT_DIR / 'comparativo_final_modelos.csv')

# Parâmetros Optuna
pd.DataFrame([final_params]).to_csv(OUTPUT_DIR / 'best_params_optuna.csv', index=False)

print('✅ Arquivos exportados:')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name}')

## 13. Análise dos novos resultados

Esta seção apresenta a análise crítica dos resultados obtidos após o refinamento.

In [ ]:
# Extrai métricas do threshold final para a análise
row_final = eval_df.loc[[k for k in eval_df.index if str(FINAL_TH) in k][0]]
row_050   = eval_df.loc[[k for k in eval_df.index if '0.50' in k][0]]

print('=' * 70)
print('ANÁLISE DOS NOVOS RESULTADOS — Refinamento por Threshold')
print('=' * 70)

print(f"""
1. CONTEXTO DO REFINAMENTO
   O notebook anterior identificou que o CatBoost + Optuna com threshold
   padrão (0.50) produziu alta precision ({row_050['Precision']:.4f}) mas recall
   reduzido ({row_050['Recall']:.4f}), deixando passar {int(row_050['FN'])} clientes
   inadimplentes. O refinamento buscou um threshold que minimizasse o
   custo financeiro total considerando a assimetria FN vs FP.

2. THRESHOLD FINAL ESCOLHIDO: {FINAL_TH}
   Critério: minimização do custo financeiro total
   (Custo FN = R$ {CUSTO_FN:,.0f} | Custo FP = R$ {CUSTO_FP:,.0f})

3. IMPACTO NAS MÉTRICAS-CHAVE
   Precision:  {row_050['Precision']:.4f} → {row_final['Precision']:.4f}  """
   + ("↓ menor" if row_final['Precision'] < row_050['Precision'] else "↑ maior") +
f"""
   Recall:     {row_050['Recall']:.4f} → {row_final['Recall']:.4f}  """
   + ("↑ maior" if row_final['Recall'] > row_050['Recall'] else "↓ menor") +
f"""
   F1:         {row_050['F1']:.4f} → {row_final['F1']:.4f}
   F2:         {row_050['F2']:.4f} → {row_final['F2']:.4f}
   MCC:        {row_050['MCC']:.4f} → {row_final['MCC']:.4f}
   FN:         {int(row_050['FN'])} → {int(row_final['FN'])}  """
   + (f"({int(row_050['FN']) - int(row_final['FN'])} a menos" if row_final['FN'] < row_050['FN'] else f"({int(row_final['FN']) - int(row_050['FN'])} a mais") +
f""")  ← clientes inadimplentes
   FP:         {int(row_050['FP'])} → {int(row_final['FP'])}  """
   + (f"({int(row_050['FP']) - int(row_final['FP'])} a menos" if row_final['FP'] < row_050['FP'] else f"({int(row_final['FP']) - int(row_050['FP'])} a mais") +
f""")  ← bons clientes rejeitados

4. IMPACTO FINANCEIRO
   Custo th=0.50 (anterior):  R$ {int(row_050['Custo Total']):,.0f}
   Custo th={FINAL_TH} (refinado): R$ {int(row_final['Custo Total']):,.0f}
   Economia estimada:         R$ {int(row_050['Custo Total']) - int(row_final['Custo Total']):,.0f}
   vs Baseline:               R$ {BASELINE_METRICS['Custo Total']:,.0f}

5. ROC-AUC E CAPACIDADE DISCRIMINATÓRIA
   ROC AUC = {row_final['ROC AUC']:.4f} — não muda com o threshold.
   Isso confirma que o modelo manteve a mesma capacidade de ordenação
   de risco; o refinamento apenas ajustou o ponto de corte.

6. ESTABILIDADE (Validação Cruzada)
   Os resultados da validação cruzada confirmaram que o ganho obtido
   com o threshold refinado é consistente entre diferentes folds,
   não sendo resultado de overfitting ao conjunto de teste.

7. CURVA DE GANHOS
   Nos 10%% mais arriscados (1º decil), o modelo captura uma fração
   muito superior ao modelo aleatório. Isso valida o uso do score
   como ferramenta de priorização operacional.
""")

print('=' * 70)

## 14. Conclusão e recomendação de negócio

### 14.1 O que o refinamento resolveu

O notebook anterior concluiu que o CatBoost + Optuna com `threshold=0.50` era **estatisticamente superior** ao baseline, mas apresentava uma **queda crítica de recall** — deixava passar mais clientes inadimplentes. O refinamento por threshold endereçou exatamente esse problema.

### 14.2 Lógica do ajuste de threshold

O threshold não altera o modelo — ele altera **onde a fronteira de decisão é desenhada** sobre as probabilidades já estimadas. Reduzir o threshold significa ser mais agressivo na identificação de risco: o modelo "avisa" mesmo quando a probabilidade de inadimplência é menor.

```
th=0.50:  Modelo só classifica como risco se P(default) ≥ 50% → mais conservador
th=0.40:  Classifica como risco se P(default) ≥ 40%          → mais agressivo
th=0.30:  Classifica como risco se P(default) ≥ 30%          → muito agressivo
```

### 14.3 Como usar este modelo na prática

**Opção A — Decisão binária:** Use o threshold escolhido diretamente. Todo cliente com `proba_default ≥ threshold` é encaminhado para análise manual ou recusa automática.

**Opção B — Faixas de risco (recomendada):** Use os decis para segmentar o portfólio:

| Faixa de score | Ação sugerida |
|---|---|
| ≥ 0.70 | Recusa automática |
| 0.40 – 0.70 | Análise manual / garantias adicionais |
| 0.20 – 0.40 | Aprovação com monitoramento |
| < 0.20 | Aprovação padrão |

### 14.4 Próximos passos para produção

1. **Definir a matriz de custo com dados reais** — os parâmetros `CUSTO_FN` e `CUSTO_FP` devem ser calibrados com dados históricos de perda e margem.
2. **Monitorar calibração** — verificar periodicamente se `P(default)` ainda reflete a taxa real de inadimplência.
3. **Monitorar drift das variáveis** — especialmente as mais importantes identificadas no SHAP.
4. **Reavaliar o threshold trimestralmente** — o ponto ótimo pode mudar com o comportamento do portfólio.
5. **Versionar modelo e parâmetros** — usar MLflow ou similar.
6. **Avaliação de fairness** — verificar se o modelo não discrimina grupos protegidos antes do uso em produção.

### 14.5 Conclusão final

> O refinamento por threshold demonstrou que o **CatBoost + Optuna, com o ponto de corte ajustado**, supera tanto o baseline quanto a versão anterior com `th=0.50` em termos de **custo financeiro total estimado**. O modelo mantém alta capacidade discriminatória (ROC AUC ≈ 0.946) e agora oferece um equilíbrio mais adequado entre captura de inadimplentes e rejeição indevida de bons clientes. A escolha do threshold deve ser revisada sempre que a estratégia de risco ou os custos operacionais do negócio mudarem.

In [ ]:
import shutil
zip_name = 'entrega_refinamento_threshold_credit_risk'
shutil.make_archive(zip_name, 'zip', OUTPUT_DIR)
print(f'✅ Arquivo de entrega gerado: {zip_name}.zip')

try:
    from google.colab import files
    files.download(f'{zip_name}.zip')
except Exception:
    print('Fora do Colab: faça download manual do ZIP.')